In [86]:
import numpy as np
import pandas as pd

DATASET_PATH = "/Users/fontana/Desktop/research/datafusion-agent/data/dataset/raw/20260403_capella_ieee_datacontest_2026_v01.csv"

# Load DataFrame
df = pd.read_csv(DATASET_PATH)

# Example: Print first few rows
df.head()

,stac_id,collect_id,datetime,start_datetime,end_datetime,center_lon,center_lat,platform,constellation,instrument_mode,...,resolution_azimuth,resolution_ground_range,pixel_spacing_range,pixel_spacing_azimuth,image_length,image_width,looks_range,looks_azimuth,orbital_plane,collection_type
0,CAPELLA_C13_SP_SLC_HH_20251105143522_202511051...,7a6bebc3-d4d6-43fb-b8b8-b2ff5f752f9f,2025-11-05T14:35:28.814850Z,2025-11-05T14:35:26.685365Z,2025-11-05T14:35:30.944336Z,-155.286305,19.421466,capella-13,capella,spotlight,...,0.44,0.27,0.25,0.41,4981.396849,10197.199779,1,1,53,spotlight
1,CAPELLA_C13_SP_SLC_HH_20251112022441_202511120...,f1e3c4b6-a64f-466a-900b-cec64250a0eb,2025-11-12T02:24:47.676882Z,2025-11-12T02:24:45.695697Z,2025-11-12T02:24:49.658068Z,-155.286169,19.420576,capella-13,capella,spotlight,...,0.44,0.28,0.26,0.41,5042.676290,10049.452929,1,1,53,spotlight
2,CAPELLA_C13_SP_GEO_HH_20251105143522_202511051...,7a6bebc3-d4d6-43fb-b8b8-b2ff5f752f9f,2025-11-05T14:35:28.814912Z,2025-11-05T14:35:22.422744Z,2025-11-05T14:35:35.207080Z,-155.286257,19.421424,capella-13,capella,spotlight,...,0.56,0.38,0.25,0.25,4981.655578,10088.212063,1,3,53,spotlight
3,CAPELLA_C13_SP_SLC_HH_20251108101220_202511081...,c2b1fe00-881f-43c9-99ed-7bad19ee492f,2025-11-08T10:12:25.262548Z,2025-11-08T10:12:23.671454Z,2025-11-08T10:12:26.853642Z,-121.870884,37.318473,capella-13,capella,spotlight,...,0.44,0.37,0.34,0.41,5010.784270,8168.277075,1,1,53,spotlight
4,CAPELLA_C13_SP_SLC_HH_20251111122850_202511111...,9d88301a-5646-4931-b81d-1129b63b2049,2025-11-11T12:28:56.582394Z,2025-11-11T12:28:54.452785Z,2025-11-11T12:28:58.712003Z,-155.286285,19.421482,capella-13,capella,spotlight,...,0.44,0.27,0.25,0.41,4981.369918,10197.050171,1,1,53,spotlight


In [87]:
import numpy as np
import pandas as pd
from typing import Tuple, Optional, List, Dict, Any

import reverse_geocoder as rg
from geopy.geocoders import Photon
from geopy.extra.rate_limiter import RateLimiter

from sklearn.cluster import DBSCAN

EARTH_RADIUS_KM = 6371.0

def spatial_clustering(X: np.ndarray, min_samples: int = 1, eps_km: float = 5.0) -> np.ndarray:
    """Applies DBSCAN clustering to spatial coordinates using the haversine metric."""
    # Calculate epsilon in radians
    eps_rad = eps_km / EARTH_RADIUS_KM

    # Apply DBSCAN with haversine metric
    spatial_dbscan = DBSCAN(eps=eps_rad, min_samples=min_samples, metric="haversine")
    
    # Fit and predict cluster labels
    clusters = spatial_dbscan.fit_predict(X)

    return clusters

def check_valid_coordinates(latitude: float, longitude: float) -> bool:
    """Checks if the provided latitude and longitude are within valid ranges."""
    return -90 <= latitude <= 90 and -180 <= longitude <= 180

def get_geocoding_context(
    coordinates: tuple,
    min_delay_seconds: float = 1.0,
) -> Dict[str, Any]:
    latitude, longitude = coordinates
    if not check_valid_coordinates(latitude, longitude):
        raise ValueError("Invalid coordinates provided.")

    geolocator = Photon(user_agent="capella-agent")
    reverse_fn = RateLimiter(geolocator.reverse, min_delay_seconds=min_delay_seconds)
    fallback_fn = rg.search

    try:
        location = reverse_fn((latitude, longitude), exactly_one=True)
        if location:
            props = location.raw.get("properties", {})
            return {
                "name": props.get("name", ""),
                "city": props.get("city", ""),
                "state": props.get("state", ""),
                "country": props.get("country", ""),
                "country_code": props.get("countrycode", ""),
            }
        else:
            return {"error": "No geocoding result found."}
    except Exception as e:
        print(f"Photon geocoding failed: {e}")

    try:
        fallback_result = fallback_fn(coordinates, mode=1)
        if fallback_result:
            return {
                "name": fallback_result[0].get("name", ""),
                "city": fallback_result[0].get("admin2", ""),
                "state": fallback_result[0].get("admin1", ""),
                "country": fallback_result[0].get("cc", ""),
                "country_code": fallback_result[0].get("cc", ""),
            }
        else:
            return {"error": "No geocoding result found in fallback."}
    except Exception as e:
        print(f"Reverse geocoder fallback failed: {e}")
        return {"error": f"All geocoding attempts failed: {e}"}

In [ ]:
# Parse datetime
df["datetime_parsed"] = pd.to_datetime(df["datetime"], utc=True)

# Drop duplicate collect_id
df = df.drop_duplicates(subset=["collect_id"], keep="first")

# Apply spatial clustering
eps_km = 5.0
min_samples = 1
coords_rad = np.radians(df[["center_lat", "center_lon"]].values)

df["spatial_cluster"] = spatial_clustering(coords_rad, eps_km=eps_km, min_samples=min_samples)

# # Summarize clusters
# cluster_summary = df.groupby("spatial_cluster").agg({
#     "datetime_parsed": ["min", "max", "count"],
#     "center_lat": "mean",
#     "center_lon": "mean"
# }).reset_index()

# Get geocoding context for each row order by spatial cluster and datetime
df = df.sort_values(by=["spatial_cluster", "datetime_parsed"])
for idx, row in df.iterrows():
    coords = (row["center_lat"], row["center_lon"])
    geocoding_info = get_geocoding_context(coords)
    df.at[idx, "geocoding_info"] = str(geocoding_info)
    print(f"Processed row {idx+1}/{len(df)}")
    print(f"Cluster: {row['spatial_cluster']}, Datetime: {row['datetime_parsed']}")
    print(f"Coordinates: {coords}, Geocoding Info: {geocoding_info}")


Processed row 1286/791
Cluster: 0, Datetime: 2024-06-04 03:56:31.270101+00:00
Coordinates: (19.405592675, -155.287365225), Geocoding Info: {'name': 'Halemaʻumaʻu Crater', 'city': '', 'state': 'Hawaii', 'country': 'United States', 'country_code': 'US'}
Processed row 1283/791
Cluster: 0, Datetime: 2024-06-05 11:29:55.778268+00:00
Coordinates: (19.404825525, -155.286569605), Geocoding Info: {'name': 'Halemaʻumaʻu Crater', 'city': '', 'state': 'Hawaii', 'country': 'United States', 'country_code': 'US'}
Processed row 1282/791
Cluster: 0, Datetime: 2024-06-07 02:44:43.722759+00:00
Coordinates: (19.40481858, -155.286490855), Geocoding Info: {'name': 'Halemaʻumaʻu Crater', 'city': '', 'state': 'Hawaii', 'country': 'United States', 'country_code': 'US'}
Processed row 1276/791
Cluster: 0, Datetime: 2024-06-10 01:32:58.937863+00:00
Coordinates: (19.405593255, -155.287363755), Geocoding Info: {'name': 'Halemaʻumaʻu Crater', 'city': '', 'state': 'Hawaii', 'country': 'United States', 'country_code':

RateLimiter caught an error, retrying (0/2 tries). Called with (*((19.39459123, -155.283144545),), **{'exactly_one': True}).
Traceback (most recent call last):
  File "/Users/fontana/Desktop/research/datafusion-agent/.venv/lib/python3.12/site-packages/urllib3/connection.py", line 204, in _new_conn
    sock = connection.create_connection(
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/fontana/Desktop/research/datafusion-agent/.venv/lib/python3.12/site-packages/urllib3/util/connection.py", line 85, in create_connection
    raise err
  File "/Users/fontana/Desktop/research/datafusion-agent/.venv/lib/python3.12/site-packages/urllib3/util/connection.py", line 73, in create_connection
    sock.connect(sa)
TimeoutError: timed out

The above exception was the direct cause of the following exception:

Traceback (most recent call last):
  File "/Users/fontana/Desktop/research/datafusion-agent/.venv/lib/python3.12/site-packages/urllib3/connectionpool.py", line 787, in urlopen
    respons

Processed row 886/791
Cluster: 0, Datetime: 2025-01-19 13:19:29.413458+00:00
Coordinates: (19.39459123, -155.283144545), Geocoding Info: {'name': 'Halemaʻumaʻu Trail', 'city': '', 'state': 'Hawaii', 'country': 'United States', 'country_code': 'US'}
Processed row 880/791
Cluster: 0, Datetime: 2025-01-19 21:55:29.071371+00:00
Coordinates: (19.42142376, -155.286257205), Geocoding Info: {'name': 'Crater Rim Trail', 'city': '', 'state': 'Hawaii', 'country': 'United States', 'country_code': 'US'}
Processed row 879/791
Cluster: 0, Datetime: 2025-01-20 11:51:20.843921+00:00
Coordinates: (19.42107284, -155.28695302), Geocoding Info: {'name': 'Crater Rim Trail', 'city': '', 'state': 'Hawaii', 'country': 'United States', 'country_code': 'US'}
Processed row 881/791
Cluster: 0, Datetime: 2025-01-20 23:25:49.497770+00:00
Coordinates: (19.408567065, -155.27013716), Geocoding Info: {'name': 'Kilauea Caldera', 'city': '', 'state': 'Hawaii', 'country': 'United States', 'country_code': 'US'}
Processed ro

KeyboardInterrupt: 

In [ ]:
from sklearn.cluster import DBSCAN

def identify_sequences(
    df: pd.DataFrame,
    spatial_eps_km: float = 5.0,
    incidence_angle_threshold: float = 3.0,
    min_sequence_length: int = 2,
) -> pd.DataFrame:
    """
    Identify temporal acquisition sequences in a Capella SAR dataset.

    Sequences are groups of acquisitions taken from the same spatial location,
    orbital plane, orbit state, observation direction, and similar incidence angle.
    Spatial clustering uses DBSCAN with the haversine metric. Within each spatial
    group, a second DBSCAN pass clusters acquisitions by incidence angle. Rows that
    belong to a valid sequence receive ``sequence_id``, ``sequence_order``, and
    ``sequence_length`` values; all other rows have NaN / None for those columns.

    Multiple product types (e.g. SLC and GEO) sharing the same ``collect_id``
    are treated as a single acquisition for sequencing purposes. Both rows
    receive identical sequence metadata.

    Parameters
    ----------
    df : pd.DataFrame
        Input dataframe. Must contain columns: ``collect_id``, ``datetime``,
        ``center_lat``, ``center_lon``, ``orbital_plane``, ``orbit_state``,
        ``observation_direction``, ``incidence_angle``.
    spatial_eps_km : float
        Spatial neighbourhood radius in kilometres for the haversine DBSCAN.
    incidence_angle_threshold : float
        Maximum incidence-angle difference (degrees) within one sequence.
    min_sequence_length : int
        Minimum number of acquisitions required to form a sequence.

    Returns
    -------
    pd.DataFrame
        Copy of *df* with three additional columns:
        ``sequence_id`` (str | None), ``sequence_order`` (float),
        ``sequence_length`` (float).
    """
    # --- Deduplicate to one row per collect_id for clustering ---
    dedup = df.drop_duplicates(subset="collect_id", keep="first").copy()

    dedup["datetime_parsed"] = pd.to_datetime(dedup["datetime"], utc=True)

    # --- Spatial clustering (haversine DBSCAN) ---
    earth_radius_km = 6371.0
    eps_rad = spatial_eps_km / earth_radius_km
    coords_rad = np.radians(dedup[["center_lat", "center_lon"]].values)
    spatial_dbscan = DBSCAN(eps=eps_rad, min_samples=1, metric="haversine")
    dedup["spatial_cluster"] = spatial_dbscan.fit_predict(coords_rad)

    # --- Sequence detection ---
    dedup["sequence_id"] = None
    dedup["sequence_order"] = np.nan
    dedup["sequence_length"] = np.nan

    seq_counter = 0
    group_keys = ["spatial_cluster", "orbital_plane", "orbit_state", "observation_direction"]

    for _, geo_group in dedup.groupby(group_keys):
        inc_angles = geo_group["incidence_angle"].values.reshape(-1, 1)
        inc_dbscan = DBSCAN(
            eps=incidence_angle_threshold,
            min_samples=min_sequence_length,
        )
        inc_dbscan.fit(inc_angles)

        for inc_label in set(inc_dbscan.labels_):
            if inc_label == -1:
                continue
            mask = inc_dbscan.labels_ == inc_label
            seq_indices = geo_group.index[mask]
            if len(seq_indices) < min_sequence_length:
                continue

            seq_counter += 1
            seq_id = f"SEQ_{seq_counter:04d}"
            sub = dedup.loc[seq_indices].sort_values("datetime_parsed")
            for rank, idx in enumerate(sub.index, start=1):
                dedup.at[idx, "sequence_id"] = seq_id
                dedup.at[idx, "sequence_order"] = rank
                dedup.at[idx, "sequence_length"] = len(seq_indices)

    # --- Map sequence info back to all rows (including GEO + SLC duplicates) ---
    seq_map = dedup.set_index("collect_id")[["sequence_id", "sequence_order", "sequence_length"]]
    result = df.copy()
    result = result.merge(seq_map, on="collect_id", how="left")

    return result

# Identify sequences and add relevant columns to the DataFrame
df_seq = identify_sequences(df)

# Order by sequence_id and datetime for better readability
df_seq = df_seq.sort_values(by=["sequence_id", "datetime"])

df_seq[["sequence_id", "stac_id","sequence_order", "sequence_length"]].head(50)


,sequence_id,stac_id,sequence_order,sequence_length
1254,SEQ_0001,CAPELLA_C10_SP_GEO_HH_20240627162715_202406271...,1.0,2.0
1252,SEQ_0001,CAPELLA_C10_SP_SLC_HH_20240627162715_202406271...,1.0,2.0
880,SEQ_0001,CAPELLA_C10_SP_SLC_HH_20250120232546_202501202...,2.0,2.0
873,SEQ_0001,CAPELLA_C10_SP_GEO_HH_20250120232546_202501202...,2.0,2.0
1285,SEQ_0002,CAPELLA_C14_SP_GEO_HH_20240604035617_202406040...,1.0,9.0
1284,SEQ_0002,CAPELLA_C14_SP_SLC_HH_20240604035617_202406040...,1.0,9.0
1281,SEQ_0002,CAPELLA_C14_SP_GEO_HH_20240607024429_202406070...,2.0,9.0
1278,SEQ_0002,CAPELLA_C14_SP_SLC_HH_20240607024429_202406070...,2.0,9.0
1275,SEQ_0002,CAPELLA_C14_SP_GEO_HH_20240610013244_202406100...,3.0,9.0
1274,SEQ_0002,CAPELLA_C14_SP_SLC_HH_20240610013244_202406100...,3.0,9.0


In [ ]:
def summarize_sequences(df: pd.DataFrame) -> pd.DataFrame:
    """
    Summarize each acquisition sequence with features relevant to SAR time-series analysis.

    Parameters
    ----------
    df : pd.DataFrame
        DataFrame output from ``identify_sequences()``. Must contain columns:
        ``sequence_id``, ``sequence_order``, ``sequence_length``, ``collect_id``,
        ``datetime``, ``center_lat``, ``center_lon``, ``incidence_angle``,
        ``platform``, ``orbital_plane``, ``orbit_state``, ``observation_direction``,
        ``product_type``, ``instrument_mode``, ``look_angle``, ``azimuth``,
        ``squint_angle``.

    Returns
    -------
    pd.DataFrame
        One row per sequence with summary features.
    """
    seq_df = df.dropna(subset=["sequence_id"]).copy()
    seq_df["datetime_parsed"] = pd.to_datetime(seq_df["datetime"], utc=True)

    # Deduplicate to one row per collect within each sequence
    dedup = seq_df.drop_duplicates(subset=["sequence_id", "collect_id"]).copy()

    def _sequence_stats(grp: pd.DataFrame) -> pd.Series:
        grp = grp.sort_values("datetime_parsed")
        dt = grp["datetime_parsed"]
        n = len(grp)

        # --- Temporal features ---
        time_span = dt.max() - dt.min()
        revisit_intervals = dt.diff().dropna()
        mean_revisit = revisit_intervals.mean() if len(revisit_intervals) > 0 else pd.NaT
        median_revisit = revisit_intervals.median() if len(revisit_intervals) > 0 else pd.NaT
        min_revisit = revisit_intervals.min() if len(revisit_intervals) > 0 else pd.NaT
        max_revisit = revisit_intervals.max() if len(revisit_intervals) > 0 else pd.NaT
        std_revisit = revisit_intervals.std() if len(revisit_intervals) > 1 else pd.NaT
       
        # Regularity: coefficient of variation of revisit intervals (lower = more regular)
        revisit_seconds = revisit_intervals.dt.total_seconds()
        revisit_cv = (
            revisit_seconds.std() / revisit_seconds.mean()
            if len(revisit_seconds) > 1 and revisit_seconds.mean() > 0
            else np.nan
        )

        # --- Geometry features ---
        inc = grp["incidence_angle"]
        look = grp["look_angle"]
        azm = grp["azimuth"]
        squint = grp["squint_angle"]

        # --- Spatial footprint consistency ---
        lat_spread = grp["center_lat"].max() - grp["center_lat"].min()
        lon_spread = grp["center_lon"].max() - grp["center_lon"].min()

        # --- Acquisition context ---
        platforms = grp["platform"].unique()
        product_types = grp["product_type"].unique() if "product_type" in grp.columns else []

        return pd.Series({
            # Identity
            "sequence_length": n,
            "first_acquisition": dt.min(),
            "last_acquisition": dt.max(),
            "time_span_days": time_span.total_seconds() / 86400,

            # Temporal cadence
            "mean_revisit_days": mean_revisit.total_seconds() / 86400 if pd.notna(mean_revisit) else np.nan,
            "median_revisit_days": median_revisit.total_seconds() / 86400 if pd.notna(median_revisit) else np.nan,
            "min_revisit_days": min_revisit.total_seconds() / 86400 if pd.notna(min_revisit) else np.nan,
            "max_revisit_days": max_revisit.total_seconds() / 86400 if pd.notna(max_revisit) else np.nan,
            "std_revisit_days": std_revisit.total_seconds() / 86400 if pd.notna(std_revisit) else np.nan,
            "revisit_regularity_cv": revisit_cv,

            # Geometry stability
            "incidence_angle_mean": inc.mean(),
            "incidence_angle_std": inc.std(),
            "incidence_angle_range": inc.max() - inc.min(),
            "look_angle_mean": look.mean(),
            "look_angle_std": look.std(),
            "azimuth_mean": azm.mean(),
            "azimuth_std": azm.std(),
            "squint_angle_mean": squint.mean(),
            "squint_angle_std": squint.std(),

            # Spatial consistency
            "center_lat_mean": grp["center_lat"].mean(),
            "center_lon_mean": grp["center_lon"].mean(),
            "lat_spread_deg": lat_spread,
            "lon_spread_deg": lon_spread,

            # Acquisition context
            "orbital_plane": grp["orbital_plane"].iloc[0],
            "orbit_state": grp["orbit_state"].iloc[0],
            "observation_direction": grp["observation_direction"].iloc[0],
            "n_platforms": len(platforms),
            "platforms": ", ".join(sorted(platforms)),
            "product_types": ", ".join(sorted(set(product_types))),
        })

    summary = dedup.groupby("sequence_id").apply(_sequence_stats).reset_index()

    return summary

sequence_summary = summarize_sequences(df_seq)

sequence_summary.head()

,sequence_id,sequence_length,first_acquisition,last_acquisition,time_span_days,mean_revisit_days,median_revisit_days,min_revisit_days,max_revisit_days,std_revisit_days,...,center_lat_mean,center_lon_mean,lat_spread_deg,lon_spread_deg,orbital_plane,orbit_state,observation_direction,n_platforms,platforms,product_types
0,SEQ_0001,2,2024-06-27 16:27:30.300409+00:00,2025-01-20 23:25:49.497770+00:00,207.290500,207.290500,207.290500,207.290500,207.290500,NaN,...,19.406688,-155.278336,0.003759,0.016398,45,ascending,left,1,capella-10,"GEO, SLC"
1,SEQ_0002,9,2024-06-04 03:56:31.270101+00:00,2024-07-15 11:11:54.426959+00:00,41.302351,5.162794,2.950173,2.950098,17.701001,5.170314,...,19.404984,-155.286702,0.000814,0.000876,45,ascending,left,1,capella-14,"GEO, SLC"
2,SEQ_0003,2,2025-02-13 21:17:45.264097+00:00,2025-08-17 17:25:19.909974+00:00,184.838595,184.838595,184.838595,184.838595,184.838595,NaN,...,19.408513,-155.276279,0.000054,0.012369,45,ascending,right,1,capella-14,SLC
3,SEQ_0004,2,2024-06-05 11:29:55.778268+00:00,2024-06-12 09:04:28.936231+00:00,6.898995,6.898995,6.898995,6.898995,6.898995,NaN,...,19.404842,-155.286587,0.000033,0.000036,45,descending,left,2,"capella-10, capella-9",SLC
4,SEQ_0005,5,2024-06-14 09:05:34.885015+00:00,2024-06-26 04:18:34.225379+00:00,11.800687,2.950172,2.950169,2.950095,2.950254,0.000087,...,19.404608,-155.286729,0.000828,0.000752,45,descending,right,1,capella-14,"GEO, SLC"


In [ ]:
def get_geocoding_context(lat, lon, reverse_fn, fallback_search_fn):
    """
    Reverse geocode a single coordinate pair.

    Parameters
    ----------
    lat : float
        Latitude in decimal degrees.
    lon : float
        Longitude in decimal degrees.
    reverse_fn : callable
        Rate-limited reverse geocoding function (e.g. geopy RateLimiter).
    fallback_search_fn : callable
        Offline fallback function (e.g. reverse_geocoder.search).

    Returns
    -------
    dict
        Dictionary with 'address' and 'raw' keys.
    """
    if not (-90 <= lat <= 90) or not (-180 <= lon <= 180):
        return {"address": "Error: invalid coordinates", "raw": {}}

    try:
        location = reverse_fn((lat, lon))
        if location:
            return {"address": location.address, "raw": location.raw}
    except Exception:
        pass

    try:
        result = fallback_search_fn((lat, lon))[0]
        return {
            "address": f"{result['name']}, {result['admin1']}, {result['cc']}",
            "raw": result,
        }
    except Exception:
        return {"address": f"Unknown ({lat:.4f}, {lon:.4f})", "raw": {}}


def _parse_geocoding_result(info: dict) -> dict:
    """
    Extract structured fields from a geocoding result,
    handling both Photon and reverse_geocoder formats.
    """
    raw = info.get("raw", {})
    props = raw.get("properties", raw)

    return {
        "location_name": props.get("name", ""),
        "city": props.get("city", props.get("admin2", "")),
        "state": props.get("state", props.get("admin1", "")),
        "country": props.get("country", ""),
        "country_code": props.get("countrycode", props.get("cc", "")),
        "full_address": info.get("address", ""),
    }


def enrich_summary_with_geocoding(
    summary: pd.DataFrame,
    photon_user_agent: str = "capella-sar",
    min_delay_seconds: float = 0.5,
) -> pd.DataFrame:
    """
    Add geocoding columns to the sequence summary DataFrame.

    Parameters
    ----------
    summary : pd.DataFrame
        Output from ``summarize_sequences()``. Must contain
        ``center_lat_mean`` and ``center_lon_mean``.
    photon_user_agent : str
        User-agent string for the Photon geocoder.
    min_delay_seconds : float
        Minimum delay between Photon API calls.

    Returns
    -------
    pd.DataFrame
        Copy of *summary* with added columns: ``location_name``, ``city``,
        ``state``, ``country``, ``country_code``, ``full_address``.
    """
    from geopy.geocoders import Photon
    from geopy.extra.rate_limiter import RateLimiter
    import reverse_geocoder as rg

    geolocator = Photon(user_agent=photon_user_agent)
    reverse_fn = RateLimiter(geolocator.reverse, min_delay_seconds=min_delay_seconds)
    fallback_fn = rg.search

    result = summary.copy()
    geo_records = []

    for _, row in result.iterrows():
        info = get_geocoding_context(
            row["center_lat_mean"],
            row["center_lon_mean"],
            reverse_fn,
            fallback_fn,
        )
        geo_records.append(_parse_geocoding_result(info))

    geo_df = pd.DataFrame(geo_records, index=result.index)
    result = pd.concat([result, geo_df], axis=1)

    return result

summary_enriched = enrich_summary_with_geocoding(sequence_summary)

KeyboardInterrupt: 

In [ ]:
summary_enriched.head(20)

,sequence_id,sequence_length,first_acquisition,last_acquisition,time_span_days,mean_revisit_days,median_revisit_days,min_revisit_days,max_revisit_days,std_revisit_days,...,observation_direction,n_platforms,platforms,product_types,location_name,city,state,country,country_code,full_address
0,SEQ_0001,2,2024-06-27 16:27:30.300409+00:00,2025-01-20 23:25:49.497770+00:00,207.290500,207.290500,207.290500,207.290500,207.290500,NaN,...,left,1,capella-10,"GEO, SLC",Halemaʻumaʻu Crater,,Hawaii,United States,US,"Halemaʻumaʻu Crater, Hawaii, United States"
1,SEQ_0002,9,2024-06-04 03:56:31.270101+00:00,2024-07-15 11:11:54.426959+00:00,41.302351,5.162794,2.950173,2.950098,17.701001,5.170314,...,left,1,capella-14,"GEO, SLC",Halemaʻumaʻu Crater,,Hawaii,United States,US,"Halemaʻumaʻu Crater, Hawaii, United States"
2,SEQ_0003,2,2025-02-13 21:17:45.264097+00:00,2025-08-17 17:25:19.909974+00:00,184.838595,184.838595,184.838595,184.838595,184.838595,NaN,...,right,1,capella-14,SLC,Kīlauea,,Hawaii,United States,US,"Kīlauea, 96718, Hawaii, United States"
3,SEQ_0004,2,2024-06-05 11:29:55.778268+00:00,2024-06-12 09:04:28.936231+00:00,6.898995,6.898995,6.898995,6.898995,6.898995,NaN,...,left,2,"capella-10, capella-9",SLC,Halemaʻumaʻu Crater,,Hawaii,United States,US,"Halemaʻumaʻu Crater, Hawaii, United States"
4,SEQ_0005,5,2024-06-14 09:05:34.885015+00:00,2024-06-26 04:18:34.225379+00:00,11.800687,2.950172,2.950169,2.950095,2.950254,0.000087,...,right,1,capella-14,"GEO, SLC",Halemaʻumaʻu Crater,,Hawaii,United States,US,"Halemaʻumaʻu Crater, Hawaii, United States"
5,SEQ_0006,93,2025-01-08 16:04:25.047451+00:00,2025-11-12 02:24:47.676882+00:00,307.430817,3.341639,2.956067,2.955906,20.692586,2.065660,...,left,1,capella-13,"GEO, SLC",Crater Rim Trail,,Hawaii,United States,US,"Crater Rim Trail, 96788, Hawaii, United States"
6,SEQ_0007,7,2025-01-19 13:19:29.413458+00:00,2025-08-23 08:21:10.354912+00:00,215.792835,35.965472,4.434180,2.956065,156.671487,61.338641,...,right,1,capella-13,SLC,Halemaʻumaʻu Crater,,Hawaii,United States,US,"Halemaʻumaʻu Crater, Hawaii, United States"
7,SEQ_0008,6,2025-01-04 04:40:02.043768+00:00,2025-08-25 17:22:05.268186+00:00,233.529204,46.705841,5.912132,2.956068,201.012463,86.572860,...,left,1,capella-13,"GEO, SLC",Halemaʻumaʻu Crater,,Hawaii,United States,US,"Halemaʻumaʻu Crater, Hawaii, United States"
8,SEQ_0009,95,2025-01-08 02:08:34.973917+00:00,2025-11-11 12:28:56.582394+00:00,307.430806,3.270540,2.956065,2.955929,20.692623,1.961240,...,right,1,capella-13,"GEO, SLC",Crater Rim Trail,,Hawaii,United States,US,"Crater Rim Trail, 96788, Hawaii, United States"
9,SEQ_0010,88,2025-01-10 21:45:30.997999+00:00,2025-11-11 09:09:09.486399+00:00,304.474751,3.499710,2.956067,2.955915,20.692450,2.407802,...,right,1,capella-13,"GEO, SLC",ReadySpaces,San Jose,CA,United States,US,"ReadySpaces, 205, East Alma Avenue, 95112, Eas..."


In [ ]:
print(summary_enriched.to_string())

   sequence_id  sequence_length                first_acquisition                 last_acquisition  time_span_days  mean_revisit_days  median_revisit_days  min_revisit_days  max_revisit_days  std_revisit_days  revisit_regularity_cv  incidence_angle_mean  incidence_angle_std  incidence_angle_range  look_angle_mean  look_angle_std  azimuth_mean  azimuth_std  squint_angle_mean  squint_angle_std  center_lat_mean  center_lon_mean  lat_spread_deg  lon_spread_deg  orbital_plane orbit_state observation_direction  n_platforms                          platforms product_types                            location_name                                   city                 state        country country_code                                                                                                           full_address
0     SEQ_0001                2 2024-06-27 16:27:30.300409+00:00 2025-01-20 23:25:49.497770+00:00      207.290500         207.290500           207.290500        207.290500        2